# A very simple text prediction example
This example may help you understand how large language models are built.

In [ ]:
import numpy as np
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Input
from keras.utils import to_categorical

The neural network will be very simple. We will try to teach it the following text.

We will want the neural network, when we give it the beginning of a sentence, to complete the rest.

In [ ]:
text = "Umělá inteligence je nejlepší a nejjasnější budoucností lidstva matrix"

# Data preparation
We split the sentence into characters. Note that LLMs don't split text into characters, but into tokens. A token can be a word or part of a word.

Then we create a set of the characters used in the sentence.

In [ ]:
chars = sorted(list(set(text)))
print (chars)

Neural networks work with numbers, not characters. So we need to create dictionaries that map numbers to characters and vice versa.

In [ ]:
char_to_index = {char: i for i, char in enumerate(chars)}
index_to_char = {i: char for i, char in enumerate(chars)}

In [ ]:
char_to_index

In [ ]:
index_to_char

We turn the sentence into sequences. The sentence is chopped into sequences of length 5 characters. Each sequence will be labeled with the following character.

In [ ]:
seq_length = 5
sequences = []
labels = []
 
for i in range(len(text) - seq_length):
    seq = text[i:i+seq_length]
    label = text[i+seq_length]
    sequences.append([char_to_index[char] for char in seq])
    labels.append(char_to_index[label])

Converting to numpy arrays

In [ ]:
X = np.array(sequences)
y = np.array(labels)

Example of the first sequence.

The input characters (numbers) **Umělá** are followed by the character **' '** (space, index 0).

In [ ]:
print("Umělá")
print(X[0])

print(" ")
print(y[0])

Converting the data to categorical - the probability of a character occurring.

In [ ]:
X_train = to_categorical(X, len(chars))
Y_train = to_categorical(y, len(chars))

In [ ]:
X_train[0]

In [ ]:
Y_train[0]

# Neural network

The neural network will select which category the 5 input characters belong to.

The category name will be the predicted next character following the 5 input characters.

We will choose a SimpleRNN layer.

In [ ]:
model = Sequential()
model.add(Input(shape=(seq_length, len(chars)))) 
model.add(SimpleRNN(50, activation='tanh'))
model.add(Dense(len(chars), activation='softmax'))

Training the model

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X_train, Y_train, epochs=1000)

# Running the model
We present the model with the first 5 characters and let it generate 70 characters.
* We take the last 5 characters from the string
* We convert the characters to categorical input data
* We predict a single character
* The prediction returns probabilities of different characters
* We select the character with the highest probability
* We append the character to the end of the text and repeat everything

In [ ]:
start_seq = "Umělá"
generated_text = start_seq

for i in range(70):
    # create input from the last seq_length characters
    x = np.array([[char_to_index[char] for char in generated_text[-seq_length:]]])

    # convert to categorical
    x_input = to_categorical(x, len(chars))

    # predict next character probabilities
    prediction = model.predict(x_input, verbose=0)

    # select the most probable character index
    next_index = np.argmax(prediction)

    # convert index back to character
    next_char = index_to_char[next_index]

    # append to generated text
    generated_text += next_char

In [ ]:
print (f"Generated text: {generated_text}")